In [1]:
import pandas as pd
import numpy as np
import torch
from torch import nn
import torch.nn.functional as F
import re, os, shutil, zipfile
from transformers import AutoTokenizer, AutoModel, Trainer, TrainingArguments
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
from datasets import Dataset
from transformers import DataCollatorWithPadding
from torch.utils.data import DataLoader

# ==========================================
# 1. CẤU HÌNH CƠ BẢN VÀ CHUẨN BỊ DATA
# ==========================================
MODEL_NAME    = "../model/marbert_base"
tokenizer     = AutoTokenizer.from_pretrained(MODEL_NAME)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
device        = torch.device("cuda" if torch.cuda.is_available() else "cpu")

STANCE2ID    = {"Against": 0, "Favor": 1, "None": 2}
ID2LABEL     = {0: "Against", 1: "Favor", 2: "None"}
SENTIMENT2ID = {"Negative": 0, "Neutral": 1, "Positive": 2}
SARCASM2ID   = {"No": 0, "Yes": 1}

def clean_arabic_tweet(text):
    if not isinstance(text, str): return ""
    text = re.sub(r"http\S+|www\.\S+", "", text)
    text = re.sub(r"@\w+", "", text)
    text = re.sub(r"\u0640", "", text)
    text = re.sub(r"[\u064B-\u065F\u0670]", "", text)
    text = re.sub(r"[إأآا]", "ا", text)
    text = re.sub(r"ى", "ي", text)
    text = re.sub(r"ة", "ه", text)
    text = re.sub(r"(.)\1+", r"\1\1", text)
    return re.sub(r"\s+", " ", text.replace("#", " ")).strip()

def load_data(file_path, is_train=True):
    df = pd.read_csv(file_path, keep_default_na=False)
    for col in ["target", "text"]: df[col] = df[col].astype(str).str.strip()
    df["clean_text"] = df["text"].apply(clean_arabic_tweet)
    if is_train:
        for col in ["stance", "sentiment", "sarcasm"]: df[col] = df[col].astype(str).str.strip()
        df["label_stance"]    = df["stance"].map(STANCE2ID).fillna(2).astype(int)
        df["label_sentiment"] = df["sentiment"].map(SENTIMENT2ID).fillna(1).astype(int)
        df["label_sarcasm"]   = df["sarcasm"].map(SARCASM2ID).fillna(0).astype(int)
    else:
        # Dev set: vẫn cần label để đánh giá
        if "stance" in df.columns:
            df["stance"]      = df["stance"].astype(str).str.strip()
            df["label_stance"] = df["stance"].map(STANCE2ID).fillna(2).astype(int)
    return df

print("Đang nạp dữ liệu...")
train_df = load_data("../data/train.csv", is_train=True)
dev_df   = load_data("../data/dev.csv",   is_train=False)
print(f"Train: {len(train_df)} | Dev: {len(dev_df)}")

d:\StanceEval-2026\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Đang nạp dữ liệu...
Train: 3502 | Dev: 619


In [2]:
# ==========================================
# 2. TOKENIZER, MODEL, TRAINER
# ==========================================
def tokenize_func(examples):
    tokenized = tokenizer(
        examples["target"], examples["clean_text"],
        padding="max_length", truncation=True, max_length=128
    )
    if "label_stance" in examples:
        tokenized["labels_stance"]    = examples["label_stance"]
        tokenized["labels_sentiment"] = examples["label_sentiment"]
        tokenized["labels_sarcasm"]   = examples["label_sarcasm"]
    return tokenized

class MultiTaskMARBERT(nn.Module):
    def __init__(self, model_name):
        super().__init__()
        self.bert           = AutoModel.from_pretrained(model_name)
        h                   = self.bert.config.hidden_size
        self.dropout        = nn.Dropout(0.2)
        self.stance_head    = nn.Linear(h, 3)
        self.sentiment_head = nn.Linear(h, 3)
        self.sarcasm_head   = nn.Linear(h, 2)

    def forward(self, input_ids, attention_mask, token_type_ids=None):
        out    = self.bert(input_ids=input_ids, attention_mask=attention_mask, token_type_ids=token_type_ids)
        pooled = self.dropout(out.pooler_output)
        return self.stance_head(pooled), self.sentiment_head(pooled), self.sarcasm_head(pooled)

class MultiTaskTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels_stance    = inputs.pop("labels_stance")
        labels_sentiment = inputs.pop("labels_sentiment")
        labels_sarcasm   = inputs.pop("labels_sarcasm")

        # R-DROP: 2 forward pass với dropout mask khác nhau
        logits_s1, logits_se1, logits_sa1 = model(**inputs)
        logits_s2, logits_se2, logits_sa2 = model(**inputs)

        # CE loss chính (multi-task)
        ce_loss = (
            nn.CrossEntropyLoss(label_smoothing=0.1)(logits_s1,   labels_stance)
            + 0.1  * nn.CrossEntropyLoss()(logits_se1, labels_sentiment)
            + 0.05 * nn.CrossEntropyLoss()(logits_sa1, labels_sarcasm)
        )

        # R-DROP: symmetric KL divergence chỉ trên Stance
        # (task chính duy nhất ảnh hưởng Favg2)
        def sym_kl(l1, l2):
            p1 = F.log_softmax(l1, dim=-1)
            p2 = F.log_softmax(l2, dim=-1)
            q1 = F.softmax(l1,    dim=-1)
            q2 = F.softmax(l2,    dim=-1)
            return (F.kl_div(p1, q2, reduction="batchmean") +
                    F.kl_div(p2, q1, reduction="batchmean")) / 2.0

        kl_loss    = sym_kl(logits_s1, logits_s2)
        total_loss = ce_loss + 0.5 * kl_loss

        return (total_loss, {"logits_stance": logits_s1}) if return_outputs else total_loss

def compute_metrics(eval_pred):
    lt, lbl = eval_pred.predictions, eval_pred.label_ids
    logits  = lt[0]  if isinstance(lt,  (tuple, list)) else lt
    labels  = lbl[0] if isinstance(lbl, (tuple, list)) else lbl
    logits  = np.array(logits)
    labels  = np.array(labels)
    if logits.ndim > 2:
        logits = logits.reshape(-1, logits.shape[-1]); labels = labels.flatten()
    preds = np.argmax(logits, axis=-1)
    f_ag  = f1_score(labels, preds, labels=[0], average="macro")
    f_fav = f1_score(labels, preds, labels=[1], average="macro")
    return {"Favg2": (f_ag + f_fav) / 2.0}

print("✅ Model + Trainer đã định nghĩa.")

✅ Model + Trainer đã định nghĩa.


In [3]:
# ==========================================
# 3. SWA: CHECKPOINT AVERAGING (Cách dùng đúng với HF Trainer)
# ==========================================
#
# Cơ chế hoạt động:
#   Sau khi Trainer chạy xong N epoch (mỗi epoch lưu 1 checkpoint),
#   ta load weights của K checkpoint CUỐI CÙNG ra,
#   cộng trung bình element-wise → model nằm ở "đáy phẳng" loss surface.
#
# Tại sao KHÔNG dùng SWALR/AveragedModel ở đây:
#   - Trainer quản lý optimizer nội bộ → không thể truyền optimizer ra ngoài dễ dàng
#   - Cosine LR scheduler đã giúp các epoch cuối ở vùng LR thấp ổn định
#     → checkpoint averaging của epoch 4,5,6 tương đương SWA online
#
def swa_checkpoint_averaging(model, checkpoint_dir, k=3):
    """
    Average weights của K checkpoint cuối trong checkpoint_dir.
    Các tensor được cast về float32 khi averaging (tránh lỗi bf16 precision).
    """
    # Lấy tất cả checkpoint dirs, sort theo step tăng dần
    all_ckpts = sorted(
        [d for d in os.listdir(checkpoint_dir) if d.startswith("checkpoint-")],
        key=lambda x: int(x.split("-")[-1])
    )

    selected = all_ckpts[-k:]  # K epoch cuối

    if len(selected) == 0:
        print("  ⚠️  Không tìm thấy checkpoint nào, giữ nguyên model.")
        return model

    if len(selected) < k:
        print(f"  ⚠️  Chỉ có {len(selected)}/{k} checkpoint, vẫn tiếp tục average.")

    avg_state = None
    count     = 0

    for ckpt_name in selected:
        ckpt_path = os.path.join(checkpoint_dir, ckpt_name)
        st_path   = os.path.join(ckpt_path, "model.safetensors")
        bin_path  = os.path.join(ckpt_path, "pytorch_model.bin")

        if os.path.exists(st_path):
            from safetensors.torch import load_file
            raw = load_file(st_path, device="cpu")
        elif os.path.exists(bin_path):
            raw = torch.load(bin_path, map_location="cpu", weights_only=False)
        else:
            print(f"  ⚠️  Không tìm thấy weights trong {ckpt_name}, skip.")
            continue

        # Cast về float32 để tránh mất precision khi cộng bf16
        state = {k: v.float() for k, v in raw.items()}

        if avg_state is None:
            avg_state = state
        else:
            for key in avg_state:
                avg_state[key] = avg_state[key] + state[key]
        count += 1

    if avg_state is None or count == 0:
        print("  ⚠️  Không load được checkpoint nào.")
        return model

    # Chia trung bình
    for key in avg_state:
        avg_state[key] = avg_state[key] / count

    model.load_state_dict(avg_state, strict=True)
    print(f"  ✅ SWA: average {count} checkpoints → {selected}")
    return model

print("✅ Hàm swa_checkpoint_averaging đã sẵn sàng.")

✅ Hàm swa_checkpoint_averaging đã sẵn sàng.


In [4]:
# ==========================================
# 4. 10-FOLD CV VỚI R-DROP + SWA
# ==========================================
print("\n--- BẮT ĐẦU 10-FOLD CV: R-Drop + SWA ---")
N_SPLITS = 10
SWA_K    = 3   # Average 3 epoch cuối (epoch 4, 5, 6)

skf       = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)
strat_key = train_df["target"] + "_" + train_df["stance"]
oof_probs = np.zeros((len(train_df), 3))

cols_data = ["target", "clean_text", "label_stance", "label_sentiment", "label_sarcasm"]

for fold, (train_idx, val_idx) in enumerate(skf.split(train_df, strat_key)):
    print(f"\n🚀 FOLD {fold+1}/{N_SPLITS}...")

    fold_train_df = train_df.iloc[train_idx]
    fold_val_df   = train_df.iloc[val_idx]

    fold_train_ds = (
        Dataset.from_pandas(fold_train_df[cols_data])
        .map(tokenize_func, batched=True)
        .remove_columns(cols_data)
    )
    fold_val_ds = (
        Dataset.from_pandas(fold_val_df[cols_data])
        .map(tokenize_func, batched=True)
        .remove_columns(cols_data)
    )

    fold_dir = f"../model/rdrop_swa_fold_{fold}"
    model    = MultiTaskMARBERT(MODEL_NAME)

    args = TrainingArguments(
        output_dir                  = fold_dir,
        learning_rate               = 2e-5,
        per_device_train_batch_size = 16,
        num_train_epochs            = 6,
        lr_scheduler_type           = "cosine",
        warmup_ratio                = 0.1,
        weight_decay                = 0.01,
        bf16                        = True,
        eval_strategy               = "epoch",
        save_strategy               = "epoch",
        save_total_limit            = SWA_K,     # ← Chỉ giữ K checkpoint cuối (tiết kiệm disk)
        load_best_model_at_end      = False,     # ← Tắt: SWA thay thế best-checkpoint logic
        label_names                 = ["labels_stance", "labels_sentiment", "labels_sarcasm"],
        logging_steps               = 200,
        report_to                   = "none",
    )

    trainer = MultiTaskTrainer(
        model           = model,
        args            = args,
        train_dataset   = fold_train_ds,
        eval_dataset    = fold_val_ds,
        data_collator   = data_collator,
        compute_metrics = compute_metrics,
    )
    trainer.train()

    # ── SWA: Average K checkpoint cuối thay vì chỉ dùng best checkpoint ──
    print(f"  → Áp dụng SWA (average {SWA_K} checkpoint cuối)...")
    model = swa_checkpoint_averaging(model, fold_dir, k=SWA_K)

    # Lưu SWA model (float32, không phải bf16)
    torch.save(model.state_dict(), f"../model/rdrop_swa_fold_{fold}.pt")

    # Dọn dẹp checkpoint dirs để tiết kiệm disk
    shutil.rmtree(fold_dir, ignore_errors=True)
    print(f"  🗑️  Đã xóa checkpoint dir: {fold_dir}")

    # ── Trích xuất OOF Probs ──────────────────────────────────────
    val_loader = DataLoader(
        fold_val_ds.with_format(type="torch", columns=["input_ids", "attention_mask", "token_type_ids"]),
        batch_size=32, shuffle=False
    )
    model.eval()
    model.to(device)
    fold_val_probs = []
    with torch.no_grad():
        for batch in val_loader:
            out   = model(
                input_ids      = batch["input_ids"].to(device),
                attention_mask = batch["attention_mask"].to(device),
                token_type_ids = batch["token_type_ids"].to(device),
            )
            fold_val_probs.append(F.softmax(out[0], dim=-1).cpu().numpy())

    oof_probs[val_idx] = np.vstack(fold_val_probs)

    del model, trainer
    torch.cuda.empty_cache()

# Lưu OOF probs
np.save("../model/rdrop_swa_oof_probs.npy", oof_probs)
print("\n✅ Đã lưu: '../model/rdrop_swa_oof_probs.npy'")

# Quick OOF score
oof_preds = np.argmax(oof_probs, axis=-1)
f_ag  = f1_score(train_df["label_stance"].values, oof_preds, labels=[0], average="macro")
f_fav = f1_score(train_df["label_stance"].values, oof_preds, labels=[1], average="macro")
print(f"📊 OOF Favg2 (argmax): {(f_ag+f_fav)/2.0:.4f}")


--- BẮT ĐẦU 10-FOLD CV: R-Drop + SWA ---

🚀 FOLD 1/10...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1549.66it/s]
[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Epoch,Training Loss,Validation Loss,Favg2
1,No log,0.766940,0.806312
2,0.960728,0.715095,0.819518
3,0.710499,0.708481,0.836143
4,0.551339,0.754132,0.841673
5,0.464996,0.766919,0.837460
6,0.421784,0.761958,0.837229


  → Áp dụng SWA (average 3 checkpoint cuối)...
  ✅ SWA: average 3 checkpoints → ['checkpoint-788', 'checkpoint-985', 'checkpoint-1182']
  🗑️  Đã xóa checkpoint dir: ../model/rdrop_swa_fold_0

🚀 FOLD 2/10...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3677.92it/s]
[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


...

Epoch,Training Loss,Validation Loss,Favg2
1,No log,0.778129,0.776770
2,0.959015,0.692905,0.846347
3,0.709002,0.750349,0.815675
4,0.560219,0.780289,0.831249
5,0.472471,0.772740,0.826566
6,0.421232,0.766558,0.853162


  → Áp dụng SWA (average 3 checkpoint cuối)...
  ✅ SWA: average 3 checkpoints → ['checkpoint-788', 'checkpoint-985', 'checkpoint-1182']
  🗑️  Đã xóa checkpoint dir: ../model/rdrop_swa_fold_8

🚀 FOLD 10/10...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2070.55it/s]
[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Epoch,Training Loss,Validation Loss,Favg2
1,No log,0.828297,0.743481
2,0.961269,0.750891,0.798767
3,0.698909,0.846271,0.798770
4,0.554945,0.871642,0.797930
5,0.454941,0.881354,0.811842
6,0.412980,0.883556,0.802983


  → Áp dụng SWA (average 3 checkpoint cuối)...
  ✅ SWA: average 3 checkpoints → ['checkpoint-788', 'checkpoint-985', 'checkpoint-1182']
  🗑️  Đã xóa checkpoint dir: ../model/rdrop_swa_fold_9

✅ Đã lưu: '../model/rdrop_swa_oof_probs.npy'
📊 OOF Favg2 (argmax): 0.8244


In [5]:
# ==========================================
# 5. INFERENCE TRÊN TẬP TEST (DEV)
# ==========================================
print("\n--- INFERENCE TẬP TEST VỚI 10 SWA MODELS ---")

cols_test = ["target", "clean_text"]
test_ds = (
    Dataset.from_pandas(dev_df[cols_test])
    .map(tokenize_func, batched=True)
    .remove_columns(cols_test)
)
test_loader = DataLoader(
    test_ds.with_format(type="torch", columns=["input_ids", "attention_mask", "token_type_ids"]),
    batch_size=32, shuffle=False
)

all_fold_test_probs = []

for fold in range(N_SPLITS):
    print(f"  Fold {fold+1}: inference...")
    model = MultiTaskMARBERT(MODEL_NAME).to(device)
    model.load_state_dict(
        torch.load(f"../model/rdrop_swa_fold_{fold}.pt", map_location=device, weights_only=True)
    )
    model.eval()

    fold_test_probs = []
    with torch.no_grad():
        for batch in test_loader:
            out = model(
                input_ids      = batch["input_ids"].to(device),
                attention_mask = batch["attention_mask"].to(device),
                token_type_ids = batch["token_type_ids"].to(device),
            )
            fold_test_probs.append(F.softmax(out[0], dim=-1).cpu().numpy())

    all_fold_test_probs.append(np.vstack(fold_test_probs))
    del model
    torch.cuda.empty_cache()

final_probs = np.mean(all_fold_test_probs, axis=0)
np.save("../model/rdrop_swa_test_probs.npy", final_probs)
print("✅ Đã lưu: '../model/rdrop_swa_test_probs.npy'")

# Đánh giá trên Dev (nếu có label)
if "label_stance" in dev_df.columns:
    final_preds = np.argmax(final_probs, axis=-1)
    f_ag  = f1_score(dev_df["label_stance"].values, final_preds, labels=[0], average="macro")
    f_fav = f1_score(dev_df["label_stance"].values, final_preds, labels=[1], average="macro")
    print(f"\n📊 Dev Favg2 (R-Drop + SWA, argmax): {(f_ag+f_fav)/2.0:.4f}")
    print("   (Ref baseline: Seed42 alone = 0.8511)")


--- INFERENCE TẬP TEST VỚI 10 SWA MODELS ---


Map: 100%|██████████| 619/619 [00:00<00:00, 9652.65 examples/s]


  Fold 1: inference...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4826.28it/s]


  Fold 2: inference...


...

  Fold 9: inference...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 8906.24it/s]


  Fold 10: inference...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 10147.30it/s]


✅ Đã lưu: '../model/rdrop_swa_test_probs.npy'

📊 Dev Favg2 (R-Drop + SWA, argmax): 0.8501
   (Ref baseline: Seed42 alone = 0.8511)


In [6]:
# ==========================================
# 6. (OPTIONAL) KẾT HỢP VỚI MEGA ENSEMBLE
# ==========================================
# Nếu đã có Mega Ensemble (3 seeds × 10 folds từ notebook 07),
# có thể soft-vote thêm R-Drop+SWA probs vào để tạo Final Ensemble.

print("--- Tạo Final Ensemble: Mega (30 models) + RDrop+SWA (10 models) ---")

try:
    # Load Mega Ensemble probs (từ notebook 07)
    oof_42    = np.load("../model/marbert_oof_probs.npy")
    test_42   = np.load("../model/marbert_test_probs.npy")
    try:
        oof_1337  = np.load("../model/marbert_s1337_oof_probs.npy")
        test_1337 = np.load("../model/marbert_s1337_test_probs.npy")
        oof_2026  = np.load("../model/marbert_s2026_oof_probs.npy")
        test_2026 = np.load("../model/marbert_s2026_test_probs.npy")
        mega_oof  = (oof_42 + oof_1337 + oof_2026) / 3.0
        mega_test = (test_42 + test_1337 + test_2026) / 3.0
        has_mega  = True
        print("  ✅ Load được Mega Ensemble (3 seeds)")
    except FileNotFoundError:
        mega_oof  = oof_42
        mega_test = test_42
        has_mega  = False
        print("  ⚠️  Chỉ có seed=42, dùng seed42 thay Mega")

    rdrop_oof  = np.load("../model/rdrop_swa_oof_probs.npy")
    rdrop_test = np.load("../model/rdrop_swa_test_probs.npy")

    # Weighted blend: Mega (weight=2) + RDrop+SWA (weight=1)
    # Mega có 30 models → đáng tin cậy hơn, weight cao hơn
    final_oof  = (2 * mega_oof  + 1 * rdrop_oof)  / 3.0
    final_test = (2 * mega_test + 1 * rdrop_test) / 3.0

    # Đánh giá
    f_ag  = f1_score(train_df["label_stance"].values, np.argmax(final_oof, axis=-1), labels=[0], average="macro")
    f_fav = f1_score(train_df["label_stance"].values, np.argmax(final_oof, axis=-1), labels=[1], average="macro")
    print(f"  📊 Final OOF Favg2: {(f_ag+f_fav)/2.0:.4f}")

    if "label_stance" in dev_df.columns:
        f_ag  = f1_score(dev_df["label_stance"].values, np.argmax(final_test, axis=-1), labels=[0], average="macro")
        f_fav = f1_score(dev_df["label_stance"].values, np.argmax(final_test, axis=-1), labels=[1], average="macro")
        print(f"  📊 Final Dev Favg2: {(f_ag+f_fav)/2.0:.4f}")

    # Xuất submission
    final_preds = [ID2LABEL[p] for p in np.argmax(final_test, axis=-1)]
    with open("submission_final.txt", "w", encoding="utf-8") as f:
        f.writelines(l + "\n" for l in final_preds)
    with zipfile.ZipFile("submission_final.zip", "w", zipfile.ZIP_DEFLATED) as zf:
        zf.write("submission_final.txt")
    print("  ✅ Xuất file: 'submission_final.zip'")

except FileNotFoundError as e:
    print(f"  ⚠️  Thiếu file probs: {e}")
    print("  Hãy chạy notebook 07 trước để tạo Mega Ensemble.")

--- Tạo Final Ensemble: Mega (30 models) + RDrop+SWA (10 models) ---
  ⚠️  Chỉ có seed=42, dùng seed42 thay Mega
  📊 Final OOF Favg2: 0.8281
  📊 Final Dev Favg2: 0.8561
  ✅ Xuất file: 'submission_final.zip'
